# Report 1: final tables and figures

Reads only the CSV files saved by `wdbc/main.ipynb` and `covid/main.ipynb` (no clustering is rerun here).
Tables are saved to `data/report/`, figures (300 dpi) to `figures/report/`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib import font_manager

# Paths are relative to this notebook (Reports/Report 1/src/)
REPORT_DIR = Path("..")
WDBC_DIR = REPORT_DIR / "data" / "wdbc"
COVID_DIR = REPORT_DIR / "data" / "covid"
TAB_DIR = REPORT_DIR / "data" / "report"
FIG_DIR = REPORT_DIR / "figures" / "report"
TAB_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Common style: Times New Roman if installed, otherwise a metric-compatible serif font
SERIF = ["Times New Roman", "Liberation Serif", "Nimbus Roman", "DejaVu Serif"]
mpl.rcParams.update({
    "font.family": "serif", "font.serif": SERIF, "mathtext.fontset": "stix",
    "font.size": 10, "axes.titlesize": 10, "axes.labelsize": 10, "legend.fontsize": 9,
    "xtick.labelsize": 9, "ytick.labelsize": 9,
    "axes.formatter.use_locale": False,      # decimal point, no thousands separator
    "savefig.dpi": 300, "savefig.bbox": "tight",
})
print("font used:", font_manager.findfont(font_manager.FontProperties(family="serif")))

In [ ]:
# Table 3: WDBC, FCM c = 2, m = 2, per scaling variant (+ unscaled run from Step 9 and k-means ARI)
ev = pd.read_csv(WDBC_DIR / "wdbc_eval_main.csv")
ev = ev[ev["m"] == 2.0].set_index("variant")
km = pd.read_csv(WDBC_DIR / "wdbc_kmeans_baseline.csv")
km = km[km["method"] == "k-means"].set_index("variant")
raw = pd.read_csv(WDBC_DIR / "wdbc_scaling_area_test.csv").set_index("input").loc[["raw_all30"]]
raw.index = ["unscaled"]

t3 = pd.concat([raw[["ARI", "accuracy", "recall_B", "recall_M"]],
                ev[["ARI", "accuracy", "recall_B", "recall_M"]]])
t3["k-means ARI"] = km["ARI"].reindex(t3.index)      # no k-means run on the unscaled data -> NaN
t3[["accuracy", "recall_B", "recall_M"]] *= 100
t3 = t3.rename(index={"zscore": "z-score", "minmax": "min-max", "log1p_zscore": "log1p + z-score"},
               columns={"accuracy": "accuracy %", "recall_B": "recall benign %", "recall_M": "recall malignant %"})
t3 = t3.round({"ARI": 3, "accuracy %": 1, "recall benign %": 1, "recall malignant %": 1, "k-means ARI": 3})
t3.index.name = "scaling"
t3.to_csv(TAB_DIR / "table3_wdbc_scaling.csv")
t3

In [ ]:
# Table 4: COVID, reference variant, FCM c = 3: class means vs the paper's Table 3
STATE_NAMES = ["stabilisation", "destabilisation", "expansion"]
FEATS = ["x1", "x2", "x3", "x4"]
PAPER_T3 = pd.DataFrame({"x1": [1.57, 5.66, 13.70], "x2": [0.06, 0.39, 0.67], "x3": [3.65, 13.55, 14.23],
                         "x4": [19.64, 76.73, 183.62]}, index=STATE_NAMES)   # same values as covid/main.ipynb, Step 7

cm = pd.read_csv(COVID_DIR / "covid_fcm_c3_class_means.csv")
cm = cm[cm["variant"] == "reference"]
parts = {"paper": PAPER_T3}
for w, label in [("membership u", "ours (membership-weighted)"), ("hard", "ours (hard)")]:
    parts[label] = cm[cm["weighting"] == w].set_index("state").loc[STATE_NAMES, FEATS]
t4 = pd.concat(parts, names=["source", "state"]).round(2)
n_hard = cm[cm["weighting"] == "hard"].set_index("state").loc[STATE_NAMES, "n (hard)"].astype(int)
t4.to_csv(TAB_DIR / "table4_covid_class_means.csv")
print("country-days per state (hard):", n_hard.to_dict())
t4

In [ ]:
# Figure 1: WDBC, validity indices vs c (m = 2, best of 10 starts), from Step 6
c_sel = pd.read_csv(WDBC_DIR / "wdbc_c_selection.csv")
INDICES = {"PC": "max", "NPC": "max", "PE": "min", "NPE": "min", "XB": "min", "FS": "min"}
LABELS = {"zscore": "z-score", "minmax": "min-max", "log1p_zscore": "log1p + z-score"}

fig, axes = plt.subplots(2, 3, figsize=(7.0, 4.6), sharex=True)
for ax, (idx, best_dir) in zip(axes.ravel(), INDICES.items()):
    for name, g in c_sel.groupby("variant", sort=False):
        ax.plot(g["c"], g[idx], marker="o", markersize=3, lw=1.1, label=LABELS[name])
    if idx == "XB":
        ax.set_yscale("log")        # XB explodes when two prototypes almost coincide
    ax.set_title(f"{idx} ({best_dir} is better)")
    ax.set_xticks(sorted(c_sel["c"].unique()))
    ax.grid(alpha=0.3)
for ax in axes[-1]:
    ax.set_xlabel("number of clusters c")
axes[0, 0].legend(frameon=False)
fig.tight_layout()
fig.savefig(FIG_DIR / "figure1_wdbc_validity_indices.png")
plt.show()

In [ ]:
# Figure 2: COVID, reference variant, FCM c = 3, m = 2: memberships of the 8 countries of the paper's Figure 2
mem = pd.read_csv(COVID_DIR / "covid_fcm_c3_reference_memberships.csv", parse_dates=["date"])
FIG2_COUNTRIES = ["Poland", "Czechia", "Slovakia", "Germany", "France", "Italy", "Greece", "Spain"]
COLORS = {"stabilisation": "tab:green", "destabilisation": "tab:orange", "expansion": "tab:red"}

fig, axes = plt.subplots(4, 2, figsize=(7.0, 8.0), sharex=True, sharey=True)
for ax, country in zip(axes.ravel(), FIG2_COUNTRIES):
    g = mem[mem["countriesAndTerritories"] == country]
    for s in STATE_NAMES:
        ax.plot(g["date"], g[f"u_{s}"], color=COLORS[s], lw=1.1, label=s)
    ax.axhline(0.5, color="grey", lw=0.6, ls=":")
    ax.set_title(country)
    ax.set_ylim(-0.02, 1.02)
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
for ax in axes[:, 0]:
    ax.set_ylabel("membership")
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3, frameon=False)
fig.tight_layout(rect=(0, 0.03, 1, 1))
fig.savefig(FIG_DIR / "figure2_covid_memberships_8countries.png")
plt.show()

In [ ]:
# Figure 3: COVID, reference variant: spring FCM severity vs excess mortality, spring and autumn (Step 11)
val = pd.read_csv(COVID_DIR / "covid_validation_reference.csv", index_col=0)
corr = pd.read_csv(COVID_DIR / "covid_validation_correlations.csv")

fig, axes = plt.subplots(1, 2, figsize=(7.0, 3.6), sharex=True)
for ax, (col, title) in zip(axes, [("spring excess_per_million", "spring (4 Mar – 24 Jun 2020)"),
                                   ("autumn excess_per_million", "autumn (Oct – Dec 2020)")]):
    ok = val[["severity", col]].dropna()
    r = corr.query("variant == 'reference' and `FCM summary` == 'severity' and external == @col").iloc[0]
    ax.scatter(ok["severity"], ok[col], s=10, color="tab:blue")
    for country, row in ok.iterrows():
        ax.annotate(country.replace("_", " "), (row["severity"], row[col]), fontsize=5, alpha=0.8)
    ax.axhline(0, color="grey", lw=0.6)
    ax.set_title(f"{title}\nSpearman ρ = {r['rho']:.2f}, p = {r['p']:.4f}, n = {int(r['n'])}")
    ax.set_xlabel("FCM severity in spring")
axes[0].set_ylabel("excess deaths per million")
fig.tight_layout()
fig.savefig(FIG_DIR / "figure3_covid_severity_vs_excess.png")
plt.show()